# Skin Lesion Bias Reduction — Colab training

Runs the EfficientNetV2-B0 baseline classifier on a Colab GPU using the project code in `src/` and a Fitzpatrick17k dataset you've already uploaded.

**Order of operations**
1. Confirm GPU + mount Drive (if used)
2. Point the notebook at your code + data
3. Install dependencies
4. Train the baseline (with class weights, unfreeze schedule, save-best-by-val-loss)
5. Evaluate the best checkpoint and render a markdown bias report
6. *Optional* — kick off the cGAN training (placeholder, run later)

## 1. Verify GPU and Colab environment

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB)

import torch
print("torch:", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

Colab: True
torch: 2.10.0+cu128 cuda available: True
GPU: NVIDIA A100-SXM4-40GB
name, memory.total [MiB], memory.free [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB, 40437 MiB


In [2]:
!nvidia-smi

Sat May  2 16:21:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             47W /  400W |       6MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Connect your code and data

Two common layouts work:

- **Drive layout** — repo and dataset both sit under `MyDrive`. Mount Drive and point `PROJECT_ROOT` at the repo there. Outputs persist between sessions.
- **Local Colab layout** — clone the repo into `/content/` and put the dataset under `/content/dataset/`. Faster I/O, but everything is wiped when the runtime ends.

Edit `PROJECT_ROOT`, `DATASET_CSV`, and `IMAGE_DIR` in the cell below to match where you uploaded things.

In [3]:
# Mount Drive only if you're using the Drive layout. Skip this cell otherwise.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path

# === EDIT THESE ===
PROJECT_ROOT = Path("/content/drive/MyDrive/SkinLesionBiasReduction")  # repo root containing src/, dataset/, run_*.sh
# IMAGE_DIR    = PROJECT_ROOT / "dataset/images"
IMAGE_DIR    = PROJECT_ROOT / "dataset/images"

DATASET_CSV = PROJECT_ROOT / "dataset/fitzpatrick17k_c.csv"
# ==================

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGE_DIR:   ", IMAGE_DIR,   "exists:", IMAGE_DIR.exists())
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
assert IMAGE_DIR.exists(),    f"IMAGE_DIR does not exist:    {IMAGE_DIR}"

%cd $PROJECT_ROOT

PROJECT_ROOT: /content/drive/MyDrive/SkinLesionBiasReduction
IMAGE_DIR:    /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images exists: True
/content/drive/MyDrive/SkinLesionBiasReduction


In [5]:
import subprocess, time  
from pathlib import Path

DRIVE_IMAGE_DIR = IMAGE_DIR
LOCAL_IMAGE_DIR = Path("/content/local_images")                                                                                                                                                
LOCAL_IMAGE_DIR.mkdir(parents=True, exist_ok=True)                                                                                                                                            

def _count_files(d):
    r = subprocess.run(f"ls -1 '{d}' 2>/dev/null | wc -l",
                     shell=True, capture_output=True, text=True)
    return int(r.stdout.strip() or 0)

n_source = _count_files(DRIVE_IMAGE_DIR)
n_local  = _count_files(LOCAL_IMAGE_DIR)
print(f"Drive: {n_source} files | Local: {n_local} files "
      f"(need ~{max(0, n_source - n_local)} more)")

if n_local >= n_source > 0:
    print("Local cache is complete — skipping copy.")
else:
    # Parallel copy: ~64 concurrent cp workers amortize Drive's per-file FUSE overhead.
    # `cp -n` = skip files that already exist at the destination, so this resumes
    # from any partial copy (no rm -rf needed).
    print("Parallel-copying from Drive (64 workers)...")
    t0 = time.time()
    cmd = (
        f"cd '{DRIVE_IMAGE_DIR}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{LOCAL_IMAGE_DIR}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_local = _count_files(LOCAL_IMAGE_DIR)
    print(f"Done — {n_local} files in {time.time() - t0:.1f}s "
          f"({n_local / max(1, time.time() - t0):.0f} files/s)")
    assert n_local >= n_source, f"Copy incomplete: expected {n_source}, got {n_local}"

IMAGE_DIR = LOCAL_IMAGE_DIR
print(f"IMAGE_DIR → {IMAGE_DIR}")

Drive: 16518 files | Local: 0 files (need ~16518 more)
Parallel-copying from Drive (64 workers)...
Done — 16518 files in 151.5s (109 files/s)
IMAGE_DIR → /content/local_images


In [6]:
# Alternative: if you don't have the repo on Drive, clone it into /content/ and copy your dataset in.
# Uncomment, replace the URL with your fork, and re-run cell `configure-paths` with PROJECT_ROOT=/content/SkinLesionBiasReduction.
# !git clone git@github.com:hoangnam310/SkinLesionBiasReduction.git

In [7]:
!git fetch origin main && git reset --hard origin/main

remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 12.50 KiB | 4.00 KiB/s, done.
From https://github.com/hoangnam310/SkinLesionBiasReduction
 * branch            main       -> FETCH_HEAD
   10c21c1..a61762b  main       -> origin/main
Updating files: 100% (47/47), done.
HEAD is now at a61762b Chage the model architecture


## 3. Install dependencies

Colab images already include `torch`, `torchvision`, `numpy`, `pandas`, `Pillow`, `tqdm`, `matplotlib`, and `scipy`. The trainer additionally needs **timm** and **scikit-learn** (sklearn is usually preinstalled, timm usually is not). Tensorboard is optional and is also usually preinstalled.

In [8]:
!pip install --quiet timm 'scikit-learn>=1.3'

## 4. Train the baseline classifier

Full training at **224×224** for 40 epochs. The backbone is frozen for the first 3 epochs (head-only warmup), then unfrozen for fine-tuning at a lower LR.

Key flags being used:
- `--class_weights` — inverse-frequency weighted CrossEntropyLoss; the single biggest fix from the prior run's bias analysis.
- `--freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5` — three warmup epochs on the head, then full fine-tune.
- The trainer also auto-saves the best checkpoint by val_loss across all epochs (no extra flag needed).

> **Tip:** For a quick smoke test, set `EPOCHS = 1` and `IMAGE_SIZE = 64` to verify the pipeline works before committing to the full run.

In [9]:
IMAGE_SIZE   = 224
EPOCHS       = 40
BATCH_SIZE   = 32        # 32 fits comfortably on A100 at 224x224; use 16 on a T4.
LR           = 1e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 4

OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_efficientnet"
print("Outputs will land under:", OUTPUT_DIR)

Outputs will land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


In [10]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --image_size {IMAGE_SIZE} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --weight_decay {WEIGHT_DECAY} \
    --num_workers {NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
model.safetensors: 100% 28.8M/28.8M [00:00<00:00, 31.2MB/s]
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=0.9962, train_acc=0.5406, val_loss=0.9185, val_acc=0.6612  ← best
Epoch [2/40] train_loss=0.9038, train_acc=0.5906, val_loss=0.8865, val_acc=0.6429  ← best
Epoch [3/40] train_loss=0.8339, train_acc=0.6247, val_loss=0.8763, val_acc=0.6858  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.8009, train_acc=0.6551, val_loss=0.8578, val_acc=0.6785  ← best
Epoch [5/40] train_loss=0.7712, train_acc=0.6641, val_loss=0.8489, val_acc=0.6968  ← best
Epoch [6/40] train_loss=0.7424, train_acc=0.6852, val_loss=0.8298, val_acc=0.6959  ← best
Epoch [7/40] train_loss=0.7002, train_acc=0.7061, val_loss=0.823

## 5. Evaluate the best checkpoint

`train_baseline_efficientnet.py` restores best-by-val-loss weights before the final eval, so the auto-generated `metrics.json` already uses that snapshot. We additionally re-run `evaluate.py` to write `logs/evaluation_metrics.json` (with bias breakdowns) and then render a markdown report.

In [11]:
runs = sorted(OUTPUT_DIR.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert runs, f"No checkpoint.pt under {OUTPUT_DIR}"
LATEST_CKPT = runs[0]
print("Latest checkpoint:", LATEST_CKPT)

!python src/evaluate.py \
    --checkpoint "{LATEST_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{IMAGE_DIR}" \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"


!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

from IPython.display import Markdown, display

run_name = LATEST_CKPT.parent.name
report_path = PROJECT_ROOT / "logs" / f"{run_name}_report.md"
print("Report:", report_path)
display(Markdown(report_path.read_text()))

Latest checkpoint: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_164057/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_164057/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.7060
Macro AUROC:    0.8267
Macro AUPRC:    0.6631

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6857 macroAUROC=0.8390 macroAUPRC=0.6823
             benign: n=   55 AUROC=0.7900 AUPRC=0.4321
          malignant: n=   75 AUROC=0.8949 AUPRC=0.6986
     non-neoplastic: n=  325 AUROC=0.8323 AUPRC=0.9160
fitzpatrick_2 (n=736): acc=0.7201 macroAUROC=0.8237 macroAUPRC=0.6782
             benign: n=  102 AUROC=0.7350 AUPRC=0.4317
          malignant: n=  126 AUROC=0.8817 AUPRC=0.6884
     non-neoplastic: n=  508 AUROC=0.8544 AUPRC=0.9144
fitzpatrick_3 (n=429

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_164057/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images` |
| Generated at | 2026-05-02T16:41:20.214335 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.7060 |
| Macro AUROC | 0.8267 |
| Macro AUPRC | 0.6631 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6857 | 0.8390 | 0.6823 |
| 2 | 736 | 0.7201 | 0.8237 | 0.6782 |
| 3 | 429 | 0.6620 | 0.7948 | 0.6570 |
| 4 | 355 | 0.7521 | 0.8539 | 0.6678 |
| 5 | 160 | 0.6937 | 0.8244 | 0.6549 |
| 6 | 76 | 0.7500 | 0.8475 | 0.6160 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7900 | 75 / 0.8949 | 325 / 0.8323 |
| 2 | 102 / 0.7350 | 126 / 0.8817 | 508 / 0.8544 |
| 3 | 59 / 0.7122 | 67 / 0.8829 | 303 / 0.7893 |
| 4 | 47 / 0.8455 | 35 / 0.8859 | 273 / 0.8303 |
| 5 | 17 / 0.7577 | 17 / 0.8861 | 126 / 0.8294 |
| 6 | 9 / 0.7828 | 5 / 0.9014 | 62 / 0.8583 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.4321 | 75 / 0.6986 | 325 / 0.9160 |
| 2 | 102 / 0.4317 | 126 / 0.6884 | 508 / 0.9144 |
| 3 | 59 / 0.3825 | 67 / 0.6987 | 303 / 0.8898 |
| 4 | 47 / 0.4833 | 35 / 0.5954 | 273 / 0.9247 |
| 5 | 17 / 0.3974 | 17 / 0.6161 | 126 / 0.9511 |
| 6 | 9 / 0.3336 | 5 / 0.5475 | 62 / 0.9669 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.7060 |
| Balanced accuracy | 0.6563 |
| Macro F1 | 0.6097 |
| Weighted F1 | 0.7261 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.3305 | 0.5433 | 0.4110 | 289 |
| malignant | 0.5520 | 0.6862 | 0.6118 | 325 |
| non-neoplastic | 0.8866 | 0.7395 | 0.8064 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 157 | 33 | 99 |
| malignant | 50 | 223 | 52 |
| non-neoplastic | 268 | 148 | 1181 |


## 6. Segmentation experiments — train classifiers on cv2- and SAM2-segmented images

To isolate the effect of segmentation-aware preprocessing, run two more classifiers with the **same hyperparameters** as section 4 — only `--image_dir` changes:

- **cv2 variant** — ROI selection driven by LAB color variation, no SAM model. Fast.
- **SAM2 variant** — ROI selection constrained to SAM2's foreground mask (segmentation-aware crop).

Both are produced at **224×224** so the trainer's transform doesn't have to upscale at load time (which throws away mid-frequency texture detail).

Workflow:
1. Configure paths and segmentation knobs
2. Build the cv2-segmented dir
3. Install SAM2 + download a checkpoint
4. Build the SAM2-segmented dir
5. (Optional) Backfill SAM2 failures with cv2 so all three runs see the same md5s
6. Common training args
7. Train cv2 → train SAM2
8. Evaluate both

In [12]:
# Local-disk dirs for fast IO during training; mirrored back to Drive at the end.
LOCAL_CV2_DIR     = Path("/content/local_images_cv2_224")
LOCAL_SAM2_DIR    = Path("/content/local_images_sam2_224")
LOCAL_SAM3_DIR    = Path("/content/local_images_sam3_224")
LOCAL_MEDSAM3_DIR = Path("/content/local_images_medsam3_224")
DRIVE_CV2_DIR     = PROJECT_ROOT / "dataset/images_cv2_224"
DRIVE_SAM2_DIR    = PROJECT_ROOT / "dataset/images_sam2_224"
DRIVE_SAM3_DIR    = PROJECT_ROOT / "dataset/images_sam3_224"
DRIVE_MEDSAM3_DIR = PROJECT_ROOT / "dataset/images_medsam3_224"

# Segmentation knobs (apply to all backends)
SEG_SIZE  = 384      # SAM + ROI search resolution (downscaled to OUT_SIZE after)
OUT_SIZE  = 224      # final image side; matches the trainer's --image_size
CROP_FRAC = 0.6
MIN_SKIN  = 0.85

(PROJECT_ROOT / "logs").mkdir(parents=True, exist_ok=True)
for d in (LOCAL_CV2_DIR, LOCAL_SAM2_DIR, LOCAL_SAM3_DIR, LOCAL_MEDSAM3_DIR,
          DRIVE_CV2_DIR, DRIVE_SAM2_DIR, DRIVE_SAM3_DIR, DRIVE_MEDSAM3_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("CV2     →", DRIVE_CV2_DIR)
print("SAM2    →", DRIVE_SAM2_DIR)
print("SAM3    →", DRIVE_SAM3_DIR)
print("MedSAM3 →", DRIVE_MEDSAM3_DIR)
print("Source images at:", LOCAL_IMAGE_DIR, f"({_count_files(LOCAL_IMAGE_DIR)} files)")

CV2     → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_cv2_224
SAM2    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam2_224
SAM3    → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_sam3_224
MedSAM3 → /content/drive/MyDrive/SkinLesionBiasReduction/dataset/images_medsam3_224
Source images at: /content/local_images (16518 files)


### 6.1 cv2 segmentation (no SAM, fast)

Builds `images_cv2_224/` from the local cache. Runs in a few minutes on CPU. Resumable.

In [13]:
# !python src/preprocess_segmentation.py \
#     --backend cv2 \
#     --strategy color \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_CV2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_cv2_224.log'}"

# # Mirror to Drive so it survives a Colab disconnect
# !mkdir -p "{DRIVE_CV2_DIR}" && rsync -a "{LOCAL_CV2_DIR}/" "{DRIVE_CV2_DIR}/"
# print("cv2 dir:",
#       _count_files(LOCAL_CV2_DIR), "files local /",
#       _count_files(DRIVE_CV2_DIR), "on Drive")

### 6.2 Install SAM2 and fetch the tiny checkpoint

`sam2` is not preinstalled on Colab. The checkpoint and config name must match — `sam2_hiera_tiny.pt` pairs with `sam2_hiera_t.yaml` (the config string is resolved by name inside the sam2 package).

In [14]:
# SAM2_CKPT_DIR  = PROJECT_ROOT / "checkpoints"
# SAM2_CKPT_PATH = SAM2_CKPT_DIR / "sam2_hiera_tiny.pt"
# SAM2_CKPT_DIR.mkdir(parents=True, exist_ok=True)

# if not SAM2_CKPT_PATH.exists():
#     !curl -L -o "{SAM2_CKPT_PATH}" \
#         https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_tiny.pt
# size_mb = SAM2_CKPT_PATH.stat().st_size / 1e6 if SAM2_CKPT_PATH.exists() else 0
# print(f"SAM2 checkpoint: {SAM2_CKPT_PATH} (exists={SAM2_CKPT_PATH.exists()}, {size_mb:.1f} MB)")

# # Install SAM2 itself (Meta's repo). Skip if already installed in this runtime.
# try:
#     import sam2  # noqa: F401
#     print("sam2 already installed")
# except ImportError:
#     !pip install --quiet "git+https://github.com/facebookresearch/sam2.git"
#     import sam2  # noqa: F401
#     print("sam2 installed")

### 6.3 SAM2 segmentation (GPU)

Run the preprocessor with `--backend sam2`. ~30–45 min on a T4, ~10–15 min on an A100 for 16,519 images at `seg_size=384`. Resumable — files already in `--output_dir` are skipped.

The `tee` pipe captures the per-failure `[fail] <md5>: <exc>` lines, so you can audit which images SAM2 dropped this run.

In [15]:
# !python src/preprocess_segmentation.py \
#     --backend sam2 \
#     --strategy color \
#     --sam2_checkpoint "{SAM2_CKPT_PATH}" \
#     --sam2_config sam2_hiera_t.yaml \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

# !mkdir -p "{DRIVE_SAM2_DIR}" && rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
# print("SAM2 dir:",
#       _count_files(LOCAL_SAM2_DIR), "files local /",
#       _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.4 (Optional but recommended) Backfill SAM2 failures with cv2

`SkinLesionDataset` silently drops md5s whose `.jpg` is missing in `--image_dir`. If SAM2 fails on N images, the SAM2 trainer sees N fewer rows than the cv2 / raw trainers, and the seed-42 split produces a different cohort — breaking the comparison.

This cell re-runs the preprocessor with `--backend cv2` against the **same** `--output_dir`. Files already produced by SAM2 are skipped (resume-mode default), so this only fills in the gaps. After it finishes, `LOCAL_SAM2_DIR` should match `LOCAL_CV2_DIR`'s file count.

In [16]:
# !python src/preprocess_segmentation.py \
#     --backend cv2 \
#     --strategy color \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM2_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} 2>&1 | tee -a "{PROJECT_ROOT / 'logs' / 'preprocess_sam2_224.log'}"

# !rsync -a "{LOCAL_SAM2_DIR}/" "{DRIVE_SAM2_DIR}/"
# print("SAM2 dir after backfill:",
#       _count_files(LOCAL_SAM2_DIR), "files local /",
#       _count_files(DRIVE_SAM2_DIR), "on Drive")

### 6.5 Install SAM3 and run text-prompted segmentation

Vanilla SAM3 with the text prompt `"skin lesion"`. The Python package `sam3` is from `facebookresearch/sam3`; weights are pulled by `build_sam3_image_model()` on first call. Output schema and `--seg_size` / `--out_size` are identical to the SAM2 run, so any downstream trainer code that worked on `images_sam2_224/` works on `images_sam3_224/`.

> If `pip install` from the SAM3 GitHub URL below fails, double-check the repo URL — the package is moving and the URL may have changed.

In [17]:
# !pip uninstall numpy -y

In [18]:
# !pip install "numpy<2,>=1.26"
# try:
#     import sam3  # noqa: F401
#     print("sam3 already installed")
# except ImportError:
#     !pip install --quiet "git+https://github.com/facebookresearch/sam3.git"
#     import sam3  # noqa: F401
#     print("sam3 installed")


In [19]:
# !python src/preprocess_segmentation.py \
#     --backend sam3 \
#     --strategy color \
#     --prompt "skin lesion" \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_SAM3_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_sam3_224.log'}"

# !mkdir -p "{DRIVE_SAM3_DIR}" && rsync -a "{LOCAL_SAM3_DIR}/" "{DRIVE_SAM3_DIR}/"
# print("SAM3 dir:",
#       _count_files(LOCAL_SAM3_DIR), "files local /",
#       _count_files(DRIVE_SAM3_DIR), "on Drive")

### 6.6 Install MedSAM3 and run LoRA-fine-tuned segmentation

`Joey-S-Liu/MedSAM3` = SAM3 + LoRA, fine-tuned on medical imagery. Two prerequisites the install cell sets up:

1. The repo cloned to `/content/MedSAM3` so `infer_sam.py` (which defines `SAM3LoRAInference`) is on `PYTHONPATH`.
2. The LoRA config (`configs/full_lora_config.yaml`) and trained weights (`outputs/sam3_lora_full/best_lora_weights.pt`) present inside the repo.

The trained weights are **not** committed to the MedSAM3 repo — follow that repo's README to download them, or copy them in from Drive.

In [20]:
# MEDSAM3_DIR = Path("/content/MedSAM3")
# if not MEDSAM3_DIR.exists():
#     !git clone --depth 1 https://github.com/Joey-S-Liu/MedSAM3.git "{MEDSAM3_DIR}"
# else:
#     print(f"MedSAM3 already cloned at {MEDSAM3_DIR}")

# # Optional: install MedSAM3's Python deps if its requirements file is present.
# req = MEDSAM3_DIR / "requirements.txt"
# if req.exists():
#     !pip install --quiet -r "{req}"

# MEDSAM3_CONFIG  = MEDSAM3_DIR / "configs/full_lora_config.yaml"
# MEDSAM3_WEIGHTS = MEDSAM3_DIR / "outputs/sam3_lora_full/best_lora_weights.pt"

# # Stash the LoRA weights on Drive so you don't re-download every session.
# DRIVE_MEDSAM3_WEIGHTS = PROJECT_ROOT / "checkpoints/best_lora_weights.pt"
# if not MEDSAM3_WEIGHTS.exists() and DRIVE_MEDSAM3_WEIGHTS.exists():
#     MEDSAM3_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
#     !cp "{DRIVE_MEDSAM3_WEIGHTS}" "{MEDSAM3_WEIGHTS}"
#     print(f"Restored LoRA weights from Drive → {MEDSAM3_WEIGHTS}")

# print(f"Config:  {MEDSAM3_CONFIG} (exists={MEDSAM3_CONFIG.exists()})")
# print(f"Weights: {MEDSAM3_WEIGHTS} (exists={MEDSAM3_WEIGHTS.exists()})")
# assert MEDSAM3_CONFIG.exists(),  "Missing MedSAM3 LoRA config — see the MedSAM3 repo's README."
# assert MEDSAM3_WEIGHTS.exists(), (
#     "Missing MedSAM3 LoRA weights. Either download per the MedSAM3 README and place at "
#     f"{MEDSAM3_WEIGHTS}, or upload to {DRIVE_MEDSAM3_WEIGHTS} on Drive and rerun this cell."
# )

In [21]:
# !ls /usr/local/lib/python3.12/dist-packages/sam3/assets/

In [22]:
# !cd /usr/local/lib/python3.12/dist-packages && \
#   PYTHONPATH="{MEDSAM3_DIR}:{PROJECT_ROOT}" \
#   python "{PROJECT_ROOT}/src/preprocess_segmentation.py" \
#     --backend medsam3 \
#     --strategy color \
#     --prompt "skin lesion" \
#     --medsam3_config "{MEDSAM3_CONFIG}" \
#     --medsam3_weights "{MEDSAM3_WEIGHTS}" \
#     --medsam3_resolution 1008 \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_IMAGE_DIR}" \
#     --output_dir "{LOCAL_MEDSAM3_DIR}" \
#     --seg_size {SEG_SIZE} \
#     --out_size {OUT_SIZE} \
#     --crop_frac {CROP_FRAC} \
#     --min_skin {MIN_SKIN} \
#     --device cuda 2>&1 | tee "{PROJECT_ROOT / 'logs' / 'preprocess_medsam3_224.log'}"

# !mkdir -p "{DRIVE_MEDSAM3_DIR}" && rsync -a "{LOCAL_MEDSAM3_DIR}/" "{DRIVE_MEDSAM3_DIR}/"
# print("MedSAM3 dir:",
#       _count_files(LOCAL_MEDSAM3_DIR), "files local /",
#       _count_files(DRIVE_MEDSAM3_DIR), "on Drive")

In [23]:
# Copy segmented images from Drive → local disk for fast IO
import subprocess, time

copies = [
    ("cv2",     DRIVE_CV2_DIR,     LOCAL_CV2_DIR),
    ("SAM2",    DRIVE_SAM2_DIR,    LOCAL_SAM2_DIR),
    ("MedSAM3", DRIVE_MEDSAM3_DIR, LOCAL_MEDSAM3_DIR),
]

for label, src, dst in copies:
    dst.mkdir(parents=True, exist_ok=True)
    n_src = _count_files(src)
    n_dst = _count_files(dst)
    if n_src == 0:
        print(f"[{label}] ⚠️  Drive dir empty ({src}), skipping.")
        continue
    if n_dst >= n_src:
        print(f"[{label}] ✓ Local cache complete ({n_dst} files).")
        continue
    print(f"[{label}] Copying {n_src} files from Drive → {dst} ...")
    t0 = time.time()
    cmd = (
        f"cd '{src}' && "
        f"find . -maxdepth 1 -type f -print0 | "
        f"xargs -0 -n 64 -P 64 cp -n -t '{dst}/'"
    )
    subprocess.run(cmd, shell=True, check=True)
    n_dst = _count_files(dst)
    print(f"[{label}] Done — {n_dst} files in {time.time() - t0:.1f}s "
          f"({n_dst / max(1, time.time() - t0):.0f} files/s)")


[cv2] Copying 11059 files from Drive → /content/local_images_cv2_224 ...
[cv2] Done — 11058 files in 93.0s (119 files/s)
[SAM2] Copying 11059 files from Drive → /content/local_images_sam2_224 ...
[SAM2] Done — 11058 files in 88.6s (125 files/s)
[MedSAM3] Copying 11059 files from Drive → /content/local_images_medsam3_224 ...
[MedSAM3] Done — 11058 files in 86.2s (128 files/s)


### 6.7 Common training args

These match the section 4 baseline run (`20260427_055032`) **exactly**. The only thing that changes between the three runs is `--image_dir`.

In [24]:
# Mirror section 4 — keep these in sync if you change section 4.
SEG_IMAGE_SIZE   = 224
SEG_EPOCHS       = 40
SEG_BATCH_SIZE   = 32
SEG_LR           = 1e-4
SEG_WEIGHT_DECAY = 1e-4
SEG_NUM_WORKERS  = 4
SEG_OUTPUT_DIR   = OUTPUT_DIR     # all three runs share this parent; each gets its own timestamped subdir
print("Outputs land under:", SEG_OUTPUT_DIR)

Outputs land under: /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet


### 6.8 Train classifier on cv2-segmented images

Identical hyperparameters to the section 4 baseline (`20260427_055032`); only `--image_dir` changes.

In [25]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0242, train_acc=0.5193, val_loss=0.9618, val_acc=0.6037  ← best
Epoch [2/40] train_loss=0.9473, train_acc=0.5590, val_loss=0.9436, val_acc=0.5662  ← best
Epoch [3/40] train_loss=0.8957, train_acc=0.5819, val_loss=0.9206, val_acc=0.5854  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.8650, train_acc=0.6089, val_loss=0.9075, val_acc=0.6000  ← best
Epoch [5/40] train_loss=0.8438, train_acc=0.6219, val_loss=0.9085, val_acc=0.6137
Epoch [6/40] train_loss=0.8233, train_acc=0.6280, val_loss=0.9019, val_acc=0.6429  ← best
Epoch [7/40] train_loss=0.7925, train_acc=0.6527, val_loss=0.8991, val_acc=0.6374  ← best
Epoch [8/40] train_loss=0.7644, train_acc=

### 6.9 Train classifier on SAM2-segmented images

Same args as above, just `--image_dir` points at `LOCAL_SAM2_DIR`.

In [26]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0225, train_acc=0.5243, val_loss=0.9593, val_acc=0.5982  ← best
Epoch [2/40] train_loss=0.9477, train_acc=0.5645, val_loss=0.9389, val_acc=0.5863  ← best
Epoch [3/40] train_loss=0.9020, train_acc=0.5823, val_loss=0.9326, val_acc=0.6274  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.8682, train_acc=0.6044, val_loss=0.9239, val_acc=0.6338  ← best
Epoch [5/40] train_loss=0.8443, train_acc=0.6207, val_loss=0.9200, val_acc=0.6393  ← best
Epoch [6/40] train_loss=0.8282, train_acc=0.6298, val_loss=0.9151, val_acc=0.6447  ← best
Epoch [7/40] train_loss=0.7935, train_acc=0.6513, val_loss=0.9148, val_acc=0.6457  ← best
Epoch [8/40] train_loss=0.7691, tr

### 6.10 Train classifier on SAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_SAM3_DIR`.

In [27]:
# !python src/train_baseline_efficientnet.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{LOCAL_SAM3_DIR}" \
#     --image_size {SEG_IMAGE_SIZE} \
#     --epochs {SEG_EPOCHS} \
#     --batch_size {SEG_BATCH_SIZE} \
#     --lr {SEG_LR} \
#     --weight_decay {SEG_WEIGHT_DECAY} \
#     --num_workers {SEG_NUM_WORKERS} \
#     --class_weights \
#     --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
#     --output_dir "{SEG_OUTPUT_DIR}" \
#     --device cuda

### 6.11 Train classifier on MedSAM3-segmented images

Same args as the SAM2 run, just `--image_dir` points at `LOCAL_MEDSAM3_DIR`.

In [28]:
!python src/train_baseline_efficientnet.py \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --image_size {SEG_IMAGE_SIZE} \
    --epochs {SEG_EPOCHS} \
    --batch_size {SEG_BATCH_SIZE} \
    --lr {SEG_LR} \
    --weight_decay {SEG_WEIGHT_DECAY} \
    --num_workers {SEG_NUM_WORKERS} \
    --class_weights \
    --freeze_backbone --unfreeze_epoch 3 --fine_tune_lr 1e-5 \
    --output_dir "{SEG_OUTPUT_DIR}" \
    --device cuda

Using device: cuda
Image size: 224x224
Train: 7752 samples (243 batches), Val: 1095 (35), Test: 2211 (70)
Pretrained: True
Freeze backbone: True
Training classifier head only (backbone frozen).
Class weights (benign, malignant, non-neoplastic): [2.5136, 2.2607, 0.463]
Epoch [1/40] train_loss=1.0233, train_acc=0.5172, val_loss=0.9608, val_acc=0.6137  ← best
Epoch [2/40] train_loss=0.9478, train_acc=0.5608, val_loss=0.9391, val_acc=0.5726  ← best
Epoch [3/40] train_loss=0.9012, train_acc=0.5810, val_loss=0.9268, val_acc=0.5808  ← best
Backbone unfrozen at epoch 3; continuing fine-tuning with lr=1e-05
Epoch [4/40] train_loss=0.8636, train_acc=0.6047, val_loss=0.9161, val_acc=0.6119  ← best
Epoch [5/40] train_loss=0.8492, train_acc=0.6138, val_loss=0.9169, val_acc=0.6155
Epoch [6/40] train_loss=0.8272, train_acc=0.6302, val_loss=0.9140, val_acc=0.6429  ← best
Epoch [7/40] train_loss=0.7985, train_acc=0.6437, val_loss=0.9075, val_acc=0.6457  ← best
Epoch [8/40] train_loss=0.7721, train_acc=

### 6.12 Evaluate all four segmented runs

Each cell below picks the most recent run whose `args.image_dir` matches the segmented dir, then writes `logs/<run_name>_report.md` and renders it inline. Run them in order — `evaluate.py` overwrites `logs/evaluation_metrics.json` each call, but the per-run markdown report is keyed by run name so all four are kept.

In [29]:
import torch as _torch_for_seg
from IPython.display import Markdown, display

def _seg_latest_under(image_dir: Path, label: str) -> Path:
    """Pick the most recent run under SEG_OUTPUT_DIR whose args.image_dir == image_dir."""
    runs = sorted(SEG_OUTPUT_DIR.glob("*/checkpoint.pt"),
                  key=lambda p: p.stat().st_mtime, reverse=True)
    for ckpt in runs:
        try:
            args = _torch_for_seg.load(ckpt, map_location="cpu", weights_only=False).get("args", {})
        except Exception:
            continue
        if Path(args.get("image_dir", "")) == image_dir:
            print(f"[{label}] {ckpt}")
            return ckpt
    raise FileNotFoundError(f"No checkpoint found for image_dir={image_dir}")

CV2_CKPT = _seg_latest_under(LOCAL_CV2_DIR, "cv2")

!python src/evaluate.py \
    --checkpoint "{CV2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_CV2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

cv2_report = PROJECT_ROOT / "logs" / f"{CV2_CKPT.parent.name}_report.md"
print("cv2 report:", cv2_report)
display(Markdown(cv2_report.read_text()))

[cv2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_165411/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_165411/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6725
Macro AUROC:    0.7813
Macro AUPRC:    0.6028

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6505 macroAUROC=0.7876 macroAUPRC=0.6124
             benign: n=   55 AUROC=0.7045 AUPRC=0.2945
          malignant: n=   75 AUROC=0.8645 AUPRC=0.6569
     non-neoplastic: n=  325 AUROC=0.7937 AUPRC=0.8857
fitzpatrick_2 (n=736): acc=0.6549 macroAUROC=0.7810 macroAUPRC=0.6018
             benign: n=  102 AUROC=0.7125 AUPRC=0.3233
          malignant: n=  126 AUROC=0.8351 AUPRC=0.5926
     non-neoplastic: n=  508 AUROC=0.7955 AUPRC=0.8895
fitzpatrick_3 (n=429): acc=0.6387

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_165411/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_cv2_224` |
| Generated at | 2026-05-02T17:11:16.498830 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6725 |
| Macro AUROC | 0.7813 |
| Macro AUPRC | 0.6028 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6505 | 0.7876 | 0.6124 |
| 2 | 736 | 0.6549 | 0.7810 | 0.6018 |
| 3 | 429 | 0.6387 | 0.7755 | 0.6176 |
| 4 | 355 | 0.7408 | 0.7871 | 0.5883 |
| 5 | 160 | 0.7125 | 0.7561 | 0.6263 |
| 6 | 76 | 0.7632 | 0.7272 | 0.5601 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7045 | 75 / 0.8645 | 325 / 0.7937 |
| 2 | 102 / 0.7125 | 126 / 0.8351 | 508 / 0.7955 |
| 3 | 59 / 0.6782 | 67 / 0.8802 | 303 / 0.7680 |
| 4 | 47 / 0.7535 | 35 / 0.8359 | 273 / 0.7719 |
| 5 | 17 / 0.6754 | 17 / 0.8350 | 126 / 0.7579 |
| 6 | 9 / 0.6352 | 5 / 0.8000 | 62 / 0.7465 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.2945 | 75 / 0.6569 | 325 / 0.8857 |
| 2 | 102 / 0.3233 | 126 / 0.5926 | 508 / 0.8895 |
| 3 | 59 / 0.2997 | 67 / 0.6820 | 303 / 0.8711 |
| 4 | 47 / 0.3794 | 35 / 0.4868 | 273 / 0.8985 |
| 5 | 17 / 0.3509 | 17 / 0.6181 | 126 / 0.9099 |
| 6 | 9 / 0.2609 | 5 / 0.4967 | 62 / 0.9227 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6725 |
| Balanced accuracy | 0.6005 |
| Macro F1 | 0.5612 |
| Weighted F1 | 0.6942 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.2841 | 0.4464 | 0.3472 | 289 |
| malignant | 0.4858 | 0.6338 | 0.5501 | 325 |
| non-neoplastic | 0.8642 | 0.7214 | 0.7863 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 129 | 49 | 111 |
| malignant | 49 | 206 | 70 |
| non-neoplastic | 276 | 169 | 1152 |


In [30]:
SAM2_CKPT = _seg_latest_under(LOCAL_SAM2_DIR, "sam2")

!python src/evaluate.py \
    --checkpoint "{SAM2_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_SAM2_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

sam2_report = PROJECT_ROOT / "logs" / f"{SAM2_CKPT.parent.name}_report.md"
print("SAM2 report:", sam2_report)
display(Markdown(sam2_report.read_text()))

[sam2] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_170234/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_170234/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6807
Macro AUROC:    0.7864
Macro AUPRC:    0.6066

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6725 macroAUROC=0.8076 macroAUPRC=0.6313
             benign: n=   55 AUROC=0.7414 AUPRC=0.3551
          malignant: n=   75 AUROC=0.8818 AUPRC=0.6532
     non-neoplastic: n=  325 AUROC=0.7995 AUPRC=0.8855
fitzpatrick_2 (n=736): acc=0.6562 macroAUROC=0.7774 macroAUPRC=0.6010
             benign: n=  102 AUROC=0.7082 AUPRC=0.3370
          malignant: n=  126 AUROC=0.8326 AUPRC=0.5828
     non-neoplastic: n=  508 AUROC=0.7913 AUPRC=0.8832
fitzpatrick_3 (n=429): acc=0.673

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_170234/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_sam2_224` |
| Generated at | 2026-05-02T17:11:27.190568 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6807 |
| Macro AUROC | 0.7864 |
| Macro AUPRC | 0.6066 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6725 | 0.8076 | 0.6313 |
| 2 | 736 | 0.6562 | 0.7774 | 0.6010 |
| 3 | 429 | 0.6737 | 0.7939 | 0.6345 |
| 4 | 355 | 0.7380 | 0.7759 | 0.5676 |
| 5 | 160 | 0.7063 | 0.7847 | 0.6394 |
| 6 | 76 | 0.6842 | 0.7129 | 0.5691 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7414 | 75 / 0.8818 | 325 / 0.7995 |
| 2 | 102 / 0.7082 | 126 / 0.8326 | 508 / 0.7913 |
| 3 | 59 / 0.7082 | 67 / 0.8906 | 303 / 0.7828 |
| 4 | 47 / 0.7411 | 35 / 0.8296 | 273 / 0.7571 |
| 5 | 17 / 0.7121 | 17 / 0.8552 | 126 / 0.7869 |
| 6 | 9 / 0.6186 | 5 / 0.7944 | 62 / 0.7258 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.3551 | 75 / 0.6532 | 325 / 0.8855 |
| 2 | 102 / 0.3370 | 126 / 0.5828 | 508 / 0.8832 |
| 3 | 59 / 0.3212 | 67 / 0.6976 | 303 / 0.8845 |
| 4 | 47 / 0.3806 | 35 / 0.4298 | 273 / 0.8925 |
| 5 | 17 / 0.3986 | 17 / 0.5947 | 126 / 0.9248 |
| 6 | 9 / 0.2535 | 5 / 0.5284 | 62 / 0.9256 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6807 |
| Balanced accuracy | 0.6095 |
| Macro F1 | 0.5718 |
| Weighted F1 | 0.7000 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.3080 | 0.4775 | 0.3745 | 289 |
| malignant | 0.4963 | 0.6215 | 0.5519 | 325 |
| non-neoplastic | 0.8591 | 0.7295 | 0.7890 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 138 | 39 | 112 |
| malignant | 44 | 202 | 79 |
| non-neoplastic | 266 | 166 | 1165 |


In [31]:
MEDSAM3_CKPT = _seg_latest_under(LOCAL_MEDSAM3_DIR, "medsam3")

!python src/evaluate.py \
    --checkpoint "{MEDSAM3_CKPT}" \
    --csv_path "{DATASET_CSV}" \
    --image_dir "{LOCAL_MEDSAM3_DIR}" \
    --batch_size {SEG_BATCH_SIZE} \
    --num_workers {SEG_NUM_WORKERS} \
    --device cuda \
    --logs_dir "{PROJECT_ROOT / 'logs'}"

!python src/metrics_report.py \
    --json "{PROJECT_ROOT / 'logs' / 'evaluation_metrics.json'}"

medsam3_report = PROJECT_ROOT / "logs" / f"{MEDSAM3_CKPT.parent.name}_report.md"
print("medsam3 report:", medsam3_report)
display(Markdown(medsam3_report.read_text()))

[medsam3] /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_171103/checkpoint.pt
Using device: cuda
Loading checkpoint from /content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_171103/checkpoint.pt
Evaluating on 2211 samples (70 batches)
Evaluation Summary
Samples: 2211
Top-1 Accuracy: 0.6748
Macro AUROC:    0.7865
Macro AUPRC:    0.6042

Per-Fitzpatrick breakdown
----------------------------------------------------------------
fitzpatrick_1 (n=455): acc=0.6571 macroAUROC=0.8026 macroAUPRC=0.6359
             benign: n=   55 AUROC=0.7212 AUPRC=0.3418
          malignant: n=   75 AUROC=0.8797 AUPRC=0.6764
     non-neoplastic: n=  325 AUROC=0.8069 AUPRC=0.8896
fitzpatrick_2 (n=736): acc=0.6617 macroAUROC=0.7846 macroAUPRC=0.6023
             benign: n=  102 AUROC=0.7112 AUPRC=0.3088
          malignant: n=  126 AUROC=0.8477 AUPRC=0.6070
     non-neoplastic: n=  508 AUROC=0.7950 AUPRC=0.8910
fitzpatrick_3 (n=429): acc=0.

# Evaluation Report

| Field | Value |
| --- | --- |
| Checkpoint | `/content/drive/MyDrive/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260502_171103/checkpoint.pt` |
| Split | test |
| Image size | 224 |
| CSV path | `/content/drive/MyDrive/SkinLesionBiasReduction/dataset/fitzpatrick17k_c.csv` |
| Image dir | `/content/local_images_medsam3_224` |
| Generated at | 2026-05-02T17:11:37.755758 |

## Global metrics

| Metric | Value |
| --- | --- |
| Samples | 2211 |
| Top-1 Accuracy | 0.6748 |
| Macro AUROC | 0.7865 |
| Macro AUPRC | 0.6042 |

## Per-Fitzpatrick subgroup

| Fitzpatrick | n | Accuracy | Macro AUROC | Macro AUPRC |
| --- | --- | --- | --- | --- |
| 1 | 455 | 0.6571 | 0.8026 | 0.6359 |
| 2 | 736 | 0.6617 | 0.7846 | 0.6023 |
| 3 | 429 | 0.6410 | 0.7713 | 0.6062 |
| 4 | 355 | 0.7408 | 0.8091 | 0.6148 |
| 5 | 160 | 0.7125 | 0.7382 | 0.6127 |
| 6 | 76 | 0.7105 | 0.7445 | 0.5099 |

### Per-class AUROC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUROC) | malignant (n / AUROC) | non-neoplastic (n / AUROC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.7212 | 75 / 0.8797 | 325 / 0.8069 |
| 2 | 102 / 0.7112 | 126 / 0.8477 | 508 / 0.7950 |
| 3 | 59 / 0.6679 | 67 / 0.8889 | 303 / 0.7570 |
| 4 | 47 / 0.7760 | 35 / 0.8705 | 273 / 0.7808 |
| 5 | 17 / 0.6647 | 17 / 0.8297 | 126 / 0.7201 |
| 6 | 9 / 0.6783 | 5 / 0.8225 | 62 / 0.7327 |

### Per-class AUPRC by Fitzpatrick (one-vs-rest)

| Fitzpatrick | benign (n / AUPRC) | malignant (n / AUPRC) | non-neoplastic (n / AUPRC) |
| --- | --- | --- | --- |
| 1 | 55 / 0.3418 | 75 / 0.6764 | 325 / 0.8896 |
| 2 | 102 / 0.3088 | 126 / 0.6070 | 508 / 0.8910 |
| 3 | 59 / 0.2905 | 67 / 0.6591 | 303 / 0.8690 |
| 4 | 47 / 0.3905 | 35 / 0.5517 | 273 / 0.9023 |
| 5 | 17 / 0.3314 | 17 / 0.6191 | 126 / 0.8875 |
| 6 | 9 / 0.2210 | 5 / 0.3851 | 62 / 0.9236 |

## Classification metrics

| Metric | Value |
| --- | --- |
| Accuracy | 0.6748 |
| Balanced accuracy | 0.6092 |
| Macro F1 | 0.5698 |
| Weighted F1 | 0.6953 |

### Per-class

| Class | Precision | Recall | F1 | Support |
| --- | --- | --- | --- | --- |
| benign | 0.2867 | 0.4533 | 0.3512 | 289 |
| malignant | 0.5145 | 0.6554 | 0.5765 | 325 |
| non-neoplastic | 0.8567 | 0.7188 | 0.7818 | 1597 |

### Confusion matrix

| True \ Pred | benign | malignant | non-neoplastic |
| --- | --- | --- | --- |
| benign | 131 | 42 | 116 |
| malignant | 36 | 213 | 76 |
| non-neoplastic | 290 | 159 | 1148 |


## 7. Quick smoke test (optional)

If you want to verify the pipeline before committing to a full 40-epoch run, temporarily override section **4** with:

```python
IMAGE_SIZE = 64
EPOCHS     = 1
BATCH_SIZE = 128
```

Then re-run sections **4** and **5**. Once it completes without errors, restore the defaults (224 / 40 / 32) and launch the real training.

## 8. Optional — train the cGAN later

The generative side lives in `src/train.py` (vanilla cGAN; the WGAN-GP critic exists in `src/cgan.py` but the WGAN trainer hasn't been committed yet). To run on Colab once you're ready:

In [32]:
##Outputs land under outputs/<timestamp>/.
# !python src/train.py \
#     --csv_path "{DATASET_CSV}" \
#     --image_dir "{IMAGE_DIR}" \
#     --epochs 200 \
#     --batch_size 64 \
#     --lr 0.0002 \
#     --output_dir "{PROJECT_ROOT / 'outputs'}" \
#     --device cuda

In [33]:
# After (or during) cGAN training, watch losses + sample grids in TensorBoard:
# %load_ext tensorboard
# %tensorboard --logdir $PROJECT_ROOT/outputs